# 환경설정

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import koreanize_matplotlib
import seaborn as sns
from sqlalchemy import create_engine

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

engine = create_engine(os.environ["DB_URL"])
conn = engine.connect()

# 전처리

In [ ]:
DATE_FMT = "%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f"

RETENTION_START_HOUR = 24
RETENTION_WINDOW_DAY = 7

In [ ]:
def _normalize_sql(sql: str) -> str:

    return sql.replace("%%", "%")


def run_query(query: str, name: str | None = None) -> pd.DataFrame:
    result = conn.exec_driver_sql(_normalize_sql(query))
    rows = result.fetchall()
    df = pd.DataFrame(rows, columns=result.keys())
    if name:
        print(f"[{name}] rows={len(df):,}, cols={len(df.columns):,}")
    return df


def execute_many(sql: str) -> None:
    statements = [stmt.strip() for stmt in sql.split(";") if stmt.strip()]
    for stmt in statements:
        conn.exec_driver_sql(_normalize_sql(stmt))
    try:
        conn.commit()
    except Exception:
        pass
    print(f"Executed {len(statements):,} statements.")

## 데이터 전처리

### VIEW 생성 및 결측값 확인

- 이 과정에서 user_id 결측 + event_time 변환 실패 행 제거

In [ ]:
create_views_sql = """
DROP VIEW IF EXISTS v_events_signup;
CREATE VIEW v_events_signup AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time
FROM events_signup
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

DROP VIEW IF EXISTS v_events_content_start;
CREATE VIEW v_events_content_start AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id
FROM events_content_start
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

DROP VIEW IF EXISTS v_events_lesson_view;
CREATE VIEW v_events_lesson_view AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id,
    `lesson_id`  AS lesson_id
FROM events_lesson_view
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

DROP VIEW IF EXISTS v_events_lesson_complete;
CREATE VIEW v_events_lesson_complete AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id,
    `lesson_id`  AS lesson_id
FROM events_lesson_complete
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

DROP VIEW IF EXISTS v_events_content_end;
CREATE VIEW v_events_content_end AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id
FROM events_content_end
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

DROP VIEW IF EXISTS v_events_related_question_click;
CREATE VIEW v_events_related_question_click AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id`  AS content_id,
    `lesson_id`   AS lesson_id
FROM events_related_question_click
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;
"""

execute_many(create_views_sql)

### VIEW 생성 여부 검증

In [ ]:
run_query("""
SELECT 'v_events_signup'        AS view_name, COUNT(*) AS row_cnt FROM v_events_signup
UNION ALL SELECT 'v_events_content_start',           COUNT(*) FROM v_events_content_start
UNION ALL SELECT 'v_events_lesson_view',       COUNT(*) FROM v_events_lesson_view
UNION ALL SELECT 'v_events_lesson_complete',         COUNT(*) FROM v_events_lesson_complete
UNION ALL SELECT 'v_events_content_end',             COUNT(*) FROM v_events_content_end
UNION ALL SELECT 'v_events_related_question_click',  COUNT(*) FROM v_events_related_question_click;
""", "view_check")

### 결측값 확인

In [ ]:
DATE_FMT = '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f'

null_check_df = run_query(f"""
SELECT 'events_signup' AS table_name,
    COUNT(*) AS total,
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END) AS user_id_null,
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END) AS event_time_null
FROM events_signup
UNION ALL
SELECT 'events_content_start', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_content_start
UNION ALL
SELECT 'events_lesson_view', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_lesson_view
UNION ALL
SELECT 'events_lesson_complete', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_lesson_complete
UNION ALL
SELECT 'events_content_end', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_content_end
UNION ALL
SELECT 'events_related_question_click', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_related_question_click;
""", "null_check")

null_check_df

### 중복값 확인

In [ ]:
duplicate_check_df = run_query("""
SELECT 'events_signup' AS table_name,
    COUNT(*) AS total,
    COUNT(DISTINCT user_id, event_time) AS unique_cnt,
    COUNT(*) - COUNT(DISTINCT user_id, event_time) AS duplicated_cnt
FROM v_events_signup
UNION ALL
SELECT 'events_content_start', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_content_start
UNION ALL
SELECT 'events_lesson_view', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_lesson_view
UNION ALL
SELECT 'events_lesson_complete', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_lesson_complete
UNION ALL
SELECT 'events_content_end', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_content_end
UNION ALL
SELECT 'events_related_question_click', COUNT(*),
    COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_related_question_click;
""", "duplicate_check")

duplicate_check_df

- events_lesson_view : 134만번... 사용자가 사이트 새로고침 하거나 인터넷 껐다 켰다 하면서 여러 번 로깅되었을 가능성 있음
- events_related_question_click : 마찬가지...

### 이상치 1 : 시간 범위

In [ ]:
time_range_df = run_query("""
SELECT 'v_events_signup' AS view_name,
    MIN(event_time) AS min_t, MAX(event_time) AS max_t,
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END) AS future_cnt
FROM v_events_signup
UNION ALL
SELECT 'v_events_content_start', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_content_start
UNION ALL
SELECT 'v_events_lesson_view', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_lesson_view
UNION ALL
SELECT 'v_events_lesson_complete', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_lesson_complete
UNION ALL
SELECT 'v_events_content_end', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_content_end
UNION ALL
SELECT 'v_events_related_question_click', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_related_question_click;
""", "time_range")

time_range_df

- 2021년 데이터가 2건 있긴 한데 사실상 0.01%도 못 미치는 수준이니 그냥 패스?

### 이상치 2 : 가입 전 활동 (정합성)

In [ ]:
before_signup_df = run_query("""
WITH signup AS (
    SELECT user_id, MIN(event_time) AS signup_time
    FROM v_events_signup GROUP BY user_id
)
SELECT 'v_events_content_start' AS view_name,
    COUNT(*) AS before_signup_rows
FROM v_events_content_start sc
JOIN signup s ON sc.user_id = s.user_id
WHERE sc.event_time < s.signup_time
UNION ALL
SELECT 'v_events_lesson_view', COUNT(*)
FROM v_events_lesson_view l
JOIN signup s ON l.user_id = s.user_id
WHERE l.event_time < s.signup_time
UNION ALL
SELECT 'v_events_lesson_complete', COUNT(*)
FROM v_events_lesson_complete cl
JOIN signup s ON cl.user_id = s.user_id
WHERE cl.event_time < s.signup_time
UNION ALL
SELECT 'v_events_content_end', COUNT(*)
FROM v_events_content_end ec
JOIN signup s ON ec.user_id = s.user_id
WHERE ec.event_time < s.signup_time
UNION ALL
SELECT 'v_events_related_question_click', COUNT(*)
FROM v_events_related_question_click cq
JOIN signup s ON cq.user_id = s.user_id
WHERE cq.event_time < s.signup_time;
""", "before_signup")

before_signup_df

- 가입 전 콘텐츠 시작 : 11,920행
- 추측 원인 : 회원 가입 전 미리보기 / 데이터 소스 시계 동기화 차이 등

### 이상치 3 : 봇 의심 (한 유저가 너무 많은 이벤트)

- 유저별 일평균 활동량 분포 살펴보기

In [ ]:
user_stats_query = """
    SELECT
        user_id,
        COUNT(*) AS event_cnt,
        TIMESTAMPDIFF(DAY, MIN(event_time), MAX(event_time)) AS active_days,
        COUNT(*) / GREATEST(TIMESTAMPDIFF(DAY, MIN(event_time), MAX(event_time)), 1) AS daily_avg
    FROM v_events_lesson_view
    GROUP BY user_id
"""

df_user_stats = run_query(user_stats_query)

In [ ]:
print("전체 유저 수:", len(df_user_stats))
print("\n=== daily_avg 분포 ===")
print(df_user_stats['daily_avg'].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]))

print("\n=== 임계값별 봇으로 분류되는 유저 수 ===")
for threshold in [20, 30, 50, 100, 200, 500, 974]:
    bot_count = (df_user_stats['daily_avg'] >= threshold).sum()
    pct = bot_count / len(df_user_stats) * 100
    print(f"  ≥ {threshold:>4}/일 : {bot_count:>6}명 ({pct:.3f}%)")

- 전체 11.4만 명

- 974/일 이상: 8명
- 500/일 이상: 93명
- 200/일 이상: 441명

In [ ]:
top_suspects = df_user_stats[df_user_stats['daily_avg'] >= 500].sort_values('daily_avg', ascending=False)
print(f"500/일 이상 유저: {len(top_suspects)}명")
print("\n=== 상위 20명 ===")
print(top_suspects[['user_id', 'event_cnt', 'active_days', 'daily_avg']].head(20))

print("\n=== 일평균 분포 (500+) ===")
print(top_suspects['daily_avg'].describe())

- 20명 중 1명만 빼고 active_days가 0~1으로 확인됨
- 하루(또는 몇 시간) 동안 1,000번 이상 레슨에 진입했다

In [ ]:
sample_user = 'SAMPLE_USER_ID'

run_query(f"""
SELECT
    user_id,
    event_time,
    lesson_id,
    COUNT(*) AS dup_cnt
FROM v_events_lesson_view
WHERE user_id = '{sample_user}'
GROUP BY user_id, event_time, lesson_id
ORDER BY dup_cnt DESC
LIMIT 5;
""", "active0_check")

In [ ]:
THRESHOLD = 200
bot_candidates = df_user_stats[df_user_stats['daily_avg'] >= THRESHOLD].copy()
print(f"봇 후보: {len(bot_candidates)}명")

execute_many("""
DROP TABLE IF EXISTS bot_users;
CREATE TABLE bot_users (
    user_id     VARCHAR(64) NOT NULL PRIMARY KEY,
    event_cnt   INT,
    active_days INT,
    daily_avg   DECIMAL(10,2),
    detected_at DATETIME DEFAULT CURRENT_TIMESTAMP
) ENGINE=InnoDB;
""")

bot_candidates[['user_id', 'event_cnt', 'active_days', 'daily_avg']].to_sql(
    'bot_users', con=engine, if_exists='append', index=False
)

run_query("SELECT COUNT(*) AS cnt FROM bot_users", "bot_count")

# **RETENTION**

## 1. 정의

- Activation 양상 확인 시 당일 몰아 **1시간 이내 학습하는 인원**이 많은 것을 고려하여 다소 엄격히 퍼널 제한.
- 퍼널 순서 따라 완료하지 않았을 시 **이탈한 것**으로 판단.
    - 단순 서비스를 방문한 사용자가 아닌, **서비스 핵심 가치를 순차적으로 경험**하여야 의도를 분명히 가지고 들어온 **고순도의 잔존 유저**일 것이라 예상한 결과.
    - 선행 단계가 이후 단계에 **분명한 학습 동기를 주는지 판단**하기 위함. 인과관계 증명 용도.

In [ ]:
query = '''
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),
    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s ON sc.user_id = s.user_id AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),
    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc ON el.user_id = fc.user_id AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),
    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),
    retained_users AS (
        SELECT
            a.user_id,
            a.activation_time, -- 이 시간을 다음 CTE에서 쓸 수 있게 포함!
            MIN(el.event_time) AS retention_time
        FROM activation_users a
        JOIN v_events_lesson_view el
            ON a.user_id = el.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <  DATE_ADD(a.activation_time, INTERVAL 8 DAY)
        GROUP BY a.user_id, a.activation_time
    ),
    additional_completed_users AS (
        SELECT
            r.user_id,
            r.activation_time,
            MIN(cl.event_time) AS additional_complete_time
        FROM retained_users r
        JOIN v_events_lesson_complete cl
            ON r.user_id = cl.user_id
           AND cl.event_time >= r.retention_time
           AND cl.event_time < DATE_ADD(r.activation_time, INTERVAL 8 DAY)
        GROUP BY r.user_id, r.activation_time
    ),
    content_completed_users AS (
        SELECT DISTINCT
            ac.user_id
        FROM additional_completed_users ac
        JOIN v_events_content_end ec
            ON ac.user_id = ec.user_id
           AND ec.event_time >= ac.additional_complete_time
           AND ec.event_time < DATE_ADD(ac.activation_time, INTERVAL 8 DAY)
    )
SELECT
    COUNT(DISTINCT a.user_id) AS activation_users,
    COUNT(DISTINCT r.user_id) AS retention_users,
    COUNT(DISTINCT ac.user_id) AS additional_completed_users,
    COUNT(DISTINCT cc.user_id) AS content_completed_users,

    ROUND(COUNT(DISTINCT r.user_id) * 100.0 / NULLIF(COUNT(DISTINCT a.user_id), 0), 2) AS act_to_ret_pct,
    ROUND(COUNT(DISTINCT ac.user_id) * 100.0 / NULLIF(COUNT(DISTINCT r.user_id), 0), 2) AS ret_to_add_pct,
    ROUND(COUNT(DISTINCT cc.user_id) * 100.0 / NULLIF(COUNT(DISTINCT ac.user_id), 0), 2) AS add_to_end_pct
FROM activation_users a
LEFT JOIN retained_users r ON a.user_id = r.user_id
LEFT JOIN additional_completed_users ac ON r.user_id = ac.user_id
LEFT JOIN content_completed_users cc ON ac.user_id = cc.user_id;
'''

retention_funnel_df = pd.read_sql(query, engine)
retention_funnel_df

## 2. 퍼널 확인

In [ ]:
query = '''
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s ON sc.user_id = s.user_id AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc ON el.user_id = fc.user_id AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retained_users AS (
        SELECT
            a.user_id,
            a.activation_time,
            MIN(el.event_time) AS retention_time
        FROM activation_users a
        JOIN v_events_lesson_view el
            ON a.user_id = el.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <  DATE_ADD(a.activation_time, INTERVAL 8 DAY)
        GROUP BY a.user_id, a.activation_time
    ),

    additional_completed_users AS (
        SELECT
            r.user_id,
            r.activation_time,
            MIN(cl.event_time) AS additional_complete_time
        FROM retained_users r
        JOIN v_events_lesson_complete cl
            ON r.user_id = cl.user_id
           AND cl.event_time >= r.retention_time
           AND cl.event_time < DATE_ADD(r.activation_time, INTERVAL 8 DAY)
        GROUP BY r.user_id, r.activation_time
    ),

    content_completed_users AS (
        SELECT DISTINCT
            ac.user_id
        FROM additional_completed_users ac
        JOIN v_events_content_end ec
            ON ac.user_id = ec.user_id
           AND ec.event_time >= ac.additional_complete_time
           AND ec.event_time < DATE_ADD(ac.activation_time, INTERVAL 8 DAY)
    ),

    funnel AS (
        SELECT
            COUNT(DISTINCT s.user_id) AS signup_users,
            COUNT(DISTINCT fc.user_id) AS first_content_users,
            COUNT(DISTINCT fl.user_id) AS first_lesson_users,
            COUNT(DISTINCT a.user_id) AS activation_users,
            COUNT(DISTINCT r.user_id) AS retention_users,
            COUNT(DISTINCT ac.user_id) AS additional_completed_users,
            COUNT(DISTINCT cc.user_id) AS content_completed_users
        FROM signup s
        LEFT JOIN first_content fc ON s.user_id = fc.user_id
        LEFT JOIN first_lesson fl ON fc.user_id = fl.user_id
        LEFT JOIN activation_users a ON fl.user_id = a.user_id
        LEFT JOIN retained_users r ON a.user_id = r.user_id
        LEFT JOIN additional_completed_users ac ON r.user_id = ac.user_id
        LEFT JOIN content_completed_users cc ON ac.user_id = cc.user_id
    )

SELECT
    signup_users,
    first_content_users,
    first_lesson_users,
    activation_users,
    retention_users,
    additional_completed_users,
    content_completed_users,

    ROUND(first_content_users * 100.0 / NULLIF(signup_users, 0), 2) AS convert_first_content_pct,
    ROUND(first_lesson_users * 100.0 / NULLIF(first_content_users, 0), 2) AS convert_first_lesson_pct,
    ROUND(activation_users * 100.0 / NULLIF(first_lesson_users, 0), 2) AS convert_activation_pct,
    ROUND(retention_users * 100.0 / NULLIF(activation_users, 0), 2) AS convert_retention_pct,
    ROUND(additional_completed_users * 100.0 / NULLIF(retention_users, 0), 2) AS convert_additional_complete_pct,
    ROUND(content_completed_users * 100.0 / NULLIF(additional_completed_users, 0), 2) AS convert_content_complete_pct
FROM funnel;
'''

retention_summary_df = pd.read_sql(query, engine)
retention_summary_df

In [ ]:
steps = ['회원가입 완료', '콘텐츠 시작', '레슨 시작', '레슨 완강', '레슨 페이지 재진입', '추가 레슨 완강', '콘텐츠 완강']

counts = [
    retention_summary_df['signup_users'].iloc[0],
    retention_summary_df['first_content_users'].iloc[0],
    retention_summary_df['first_lesson_users'].iloc[0],
    retention_summary_df['activation_users'].iloc[0],
    retention_summary_df['retention_users'].iloc[0],
    retention_summary_df['additional_completed_users'].iloc[0],
    retention_summary_df['content_completed_users'].iloc[0]
]

rates = [
    None,
    retention_summary_df['convert_first_content_pct'].iloc[0],
    retention_summary_df['convert_first_lesson_pct'].iloc[0],
    retention_summary_df['convert_activation_pct'].iloc[0],
    retention_summary_df['convert_retention_pct'].iloc[0],
    retention_summary_df['convert_additional_complete_pct'].iloc[0],
    retention_summary_df['convert_content_complete_pct'].iloc[0]
]

plt.figure(figsize=(10, 6))

colors = sns.color_palette('Blues', len(steps))
barplot = sns.barplot(x=steps, y=counts, palette=colors)

for i, count in enumerate(counts):
    plt.text(i, count+(max(counts)*0.02), f'{int(count):,}명',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

    if i > 0 and rates[i] is not None:
        plt.text(i, count/2, f'({rates[i]}%)', ha='center', va='center',
                 color='white', fontsize=11, fontweight='bold')

ax = plt.gca()
ax.spines[['top', 'left', 'right']].set_visible(False)
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))

plt.title('리텐션 퍼널', fontsize=15, fontweight='bold')
plt.ylabel('User Count')
plt.ylim(0, max(counts)*1.15)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 3. 월별 코호트

In [ ]:
query = """
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s ON sc.user_id = s.user_id AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc ON el.user_id = fc.user_id AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    cohort_size AS (
        SELECT
            DATE_FORMAT(activation_time, '%%Y-%%m') AS cohort_month,
            COUNT(DISTINCT user_id) AS activation_users
        FROM activation_users
        GROUP BY 1
    ),

    retained_users AS (
        SELECT
            au.user_id,
            DATE_FORMAT(au.activation_time, '%%Y-%%m') AS cohort_month,
            -- 1~7일차면 1주차, 8~14일차면 2주차로 깔끔하게 계산
            FLOOR(DATEDIFF(el.event_time, au.activation_time) / 7) + 1 AS week_n
        FROM activation_users au
        JOIN v_events_lesson_view el
            ON au.user_id = el.user_id
           AND el.event_time >= DATE_ADD(au.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <= DATE_ADD(au.activation_time, INTERVAL 56 DAY)
    )

SELECT
    cs.cohort_month,
    ru.week_n,
    cs.activation_users,
    COUNT(DISTINCT ru.user_id) AS retained_users,
    ROUND(COUNT(DISTINCT ru.user_id) * 100.0 / cs.activation_users, 1) AS retention_rate
FROM cohort_size cs
LEFT JOIN retained_users ru ON cs.cohort_month = ru.cohort_month
WHERE ru.week_n IS NOT NULL
GROUP BY 1, 2, 3
ORDER BY 1, 2;
"""

cohort_retention = pd.read_sql(query, engine)
cohort_retention

In [ ]:
heatmap_data = cohort_retention.pivot(
    index='cohort_month',
    columns='week_n',
    values='retention_rate'
)

if 0 in heatmap_data.columns:
    heatmap_data = heatmap_data.drop(columns=[0])

heatmap_data.columns = [f'{int(col)}주차' for col in heatmap_data.columns]

annot_data = heatmap_data.map(lambda x: f'{x:.1f}' if pd.notna(x) else '')

plt.figure(figsize=(14, 8))

ax = sns.heatmap(
    heatmap_data,
    annot=annot_data,
    fmt='',
    cmap='Blues',
    linewidths=1,
    linecolor='white',
    cbar_kws={'label': '리텐션율 (%)', 'shrink': 0.8},
    vmin=0
)

plt.title('월별 활성화 코호트 주간 리텐션 히트맵', fontsize=18, fontweight='bold', pad=20)
plt.xlabel('활성화 후 경과 주차 (1주차 = 7일 이내 재방문)', fontweight='bold', labelpad=15)
plt.ylabel('활성화 코호트 (월)', fontweight='bold', labelpad=15)

plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
query = """
WITH
    signup AS (
        SELECT s.user_id, MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_lesson AS (
        SELECT el.user_id, MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN signup s ON el.user_id = s.user_id AND el.event_time >= s.signup_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT cl.user_id, MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retention_check AS (
        SELECT
            a.user_id,
            a.activation_time,
            MIN(el.event_time) AS first_return_time,
            TIMESTAMPDIFF(HOUR, a.activation_time, MIN(el.event_time)) AS hours_diff
        FROM activation_users a
        JOIN v_events_lesson_view el
            ON a.user_id = el.user_id
           AND el.event_time > a.activation_time
        GROUP BY a.user_id, a.activation_time
    )

SELECT
    CASE
        WHEN hours_diff < 1   THEN '1시간 이내'
        WHEN hours_diff < 24  THEN '1~24시간'
        WHEN hours_diff < 48  THEN '1~2일'
        WHEN hours_diff < 168 THEN '2~7일'
        ELSE '7일 초과'
    END AS return_timing,
    COUNT(DISTINCT user_id) AS user_count,
    ROUND(COUNT(DISTINCT user_id) / SUM(COUNT(DISTINCT user_id)) OVER () * 100, 1) AS pct
FROM retention_check
GROUP BY 1
ORDER BY MIN(hours_diff);
"""

return_timing_df = pd.read_sql(query, engine)
return_timing_df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plot_df = cohort_retention[cohort_retention['week_n'] == 1].copy()

plot_df = plot_df[plot_df['cohort_month'] >= '2022-08'].sort_values('cohort_month')

fig, ax1 = plt.subplots(figsize=(15, 7))
plt.title('월별 코호트 1주차 리텐션 추이 (Week 1 Retention)', fontsize=18, fontweight='bold', pad=25)

x = range(len(plot_df))
ax1.bar(x, plot_df['activation_users'], color='#bdd8f1', label='코호트 모수 (Total)', alpha=0.6)
ax1.bar(x, plot_df['retained_users'], color='#3667a6', label='1주차 잔존 유저 (Week 1)', alpha=0.8)

ax1.set_ylabel('유저 수 (명)', fontweight='bold', labelpad=15)
ax1.set_xticks(x)
ax1.set_xticklabels(plot_df['cohort_month'], rotation=45, ha='right')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))

ax2 = ax1.twinx()
ax2.plot(x, plot_df['retention_rate'], color='#e63946', marker='o', markersize=8,
         linewidth=2.5, label='1주차 리텐션율 (%)', zorder=5)

ax2.set_ylabel('리텐션율 (%)', fontweight='bold', labelpad=15, color='#e63946')
ax2.tick_params(axis='y', labelcolor='#e63946')
ax2.set_ylim(0, 100)

for i, rate in enumerate(plot_df['retention_rate']):
    ax2.text(i, rate + 2, f'{rate:.1f}%', ha='center', fontsize=9, fontweight='bold', color='#e63946')

try:
    drop_idx = list(plot_df['cohort_month']).index('2023-05')
    ax2.axvline(x=drop_idx, color='#fca311', linestyle='--', linewidth=2, alpha=0.8)
    ax2.text(drop_idx + 0.2, 90, '2023-05 리텐션 급락 지점', color='#fca311', fontweight='bold')
except ValueError:
    pass

ax1.spines['top'].set_visible(False)
ax2.spines['top'].set_visible(False)
ax1.grid(axis='y', linestyle='--', alpha=0.3)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', frameon=False)

plt.tight_layout()
plt.show()

- 가입 초기에는 5~60% 이상의 높은 리텐션을 보이나, 주차가 멀어질수록 10% 이하로 색이 옅어지는 양상.
    - 처음 몰입해서 듣는 유저는 많으나, 장기적으로 유지되는 유저는 많이 없는 것으로 확인.
- 2023년 초(1~3월) 가입 유저는 장기 리텐션이 견고하나, 중반 이후(5월~)에는 다소 옅어진 양상.
    - 마케팅 채널 혹은 방법, 서비스 자체적인 내부 개편이 존재했는지 확인 필요.
- 보다 오래 서비스에 잔존할 수 있도록 정기적인 개인화된 푸시 알림, 꾸준한 학습을 위한 루틴화 이벤트 등 동기부여 가능한 서비스 활성화 필요.
- 활성화 이후 잔존 일수가 길지 않은 것을 감안, 세밀한 분석을 위해 **리텐션 일수를 7일**로 확정.

# 4. 리텐션 종류별 특징
클래식, 범위, 롤링 리텐션 확인

## 4-1. 클래식 리텐션

In [ ]:
query = """
WITH
    signup AS (
        SELECT s.user_id, MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL GROUP BY s.user_id
    ),
    first_content AS (
        SELECT sc.user_id, MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s ON sc.user_id = s.user_id AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),
    first_lesson AS (
        SELECT el.user_id, MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc ON el.user_id = fc.user_id AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),
    activation_users AS (
        SELECT cl.user_id, MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),
    cohort_size AS (
        SELECT COUNT(DISTINCT user_id) AS total_activation_users
        FROM activation_users
    ),

    classic_return AS (
        SELECT DISTINCT
            a.user_id,
            DATEDIFF(el.event_time, a.activation_time) AS day_n
        FROM activation_users a
        JOIN v_events_lesson_view el
            ON a.user_id = el.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <  DATE_ADD(a.activation_time, INTERVAL 8 DAY)
    )

SELECT
    c.day_n,
    COUNT(DISTINCT c.user_id) AS retained_users,
    (SELECT total_activation_users FROM cohort_size) AS total_activation_users
FROM classic_return c
GROUP BY c.day_n
ORDER BY c.day_n;
"""

classic = pd.read_sql(query, engine)
classic['retention_rate'] = (classic['retained_users'] / classic['total_activation_users'] * 100).round(1)
classic

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick
import pandas as pd

classic_plot = (
    classic[classic['day_n'].between(0, 7)]
    .sort_values('day_n')
    .copy()
)

plt.figure(figsize=(10, 6))

x_labels = [f'Day {int(day)}' for day in classic_plot['day_n']]

colors = sns.color_palette('Blues', len(classic_plot))
barplot = sns.barplot(x=x_labels, y=classic_plot['retained_users'], palette=colors)

counts = classic_plot['retained_users'].tolist()
rates = classic_plot['retention_rate'].tolist()

for i, count in enumerate(counts):
    plt.text(i, count + (max(counts) * 0.02), f'{int(count):,}명',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

    if pd.notna(rates[i]):
        plt.text(i, count / 2, f'({rates[i]}%)', ha='center', va='center',
                 color='white', fontsize=11, fontweight='bold')

ax = plt.gca()
ax.spines[['top', 'left', 'right']].set_visible(False)
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))


plt.title('활성화 후 7일 재진입 유저 수', fontsize=15, fontweight='bold')
plt.xlabel('활성화 후 경과일', fontweight='bold', labelpad=10)
plt.ylabel('User Count')
plt.ylim(0, max(counts) * 1.15)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

- 활성화일에서 멀어질수록 재진입 유저 감소하는 양상
- 7일차 되는 날 약 **0.1%p 증가**

## 4-2. 범위 리텐션
1일 ~ 7일 사이 한 번이라도 다시 들어온 적이 있는가?

In [ ]:
query = """
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b
            ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s
            ON sc.user_id = s.user_id
           AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc
            ON el.user_id = fc.user_id
           AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl
            ON cl.user_id = fl.user_id
           AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    cohort_size AS (
        SELECT COUNT(DISTINCT user_id) AS total_activation_users
        FROM activation_users
    ),

    range_return AS (
        SELECT
            '1~7일' AS day_range,
            COUNT(DISTINCT a.user_id) AS retained_users
        FROM activation_users a
        JOIN v_events_lesson_view el
            ON a.user_id = el.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <  DATE_ADD(a.activation_time, INTERVAL 8 DAY)
    )

SELECT
    r.day_range,
    r.retained_users,
    c.total_activation_users
FROM range_return r
CROSS JOIN cohort_size c;
"""

range_ret_7 = pd.read_sql(query, engine)

range_ret_7['retention_rate'] = (
    range_ret_7['retained_users'] / range_ret_7['total_activation_users'] * 100
).round(1)

range_ret_7

## 4-3. 롤링 리텐션
7일 이후 서비스를 떠나지 않은 유저

In [ ]:
query = """
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b
            ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s
            ON sc.user_id = s.user_id
           AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc
            ON el.user_id = fc.user_id
           AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl
            ON cl.user_id = fl.user_id
           AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    cohort_size AS (
        SELECT COUNT(DISTINCT user_id) AS total_activation_users
        FROM activation_users
    ),

    rolling_return AS (
        SELECT
            '7일 이후' AS day_range,
            COUNT(DISTINCT a.user_id) AS retained_users
        FROM activation_users a
        JOIN v_events_lesson_view el
            ON a.user_id = el.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 8 DAY)
    )

SELECT
    r.day_range,
    r.retained_users,
    c.total_activation_users
FROM rolling_return r
CROSS JOIN cohort_size c;
"""

rolling_ret = pd.read_sql(query, engine)

rolling_ret['retention_rate'] = (
    rolling_ret['retained_users'] / rolling_ret['total_activation_users'] * 100
).round(1)

rolling_ret

## 4-4 리텐션 유형별 비교

In [ ]:
classic_day7_rate = classic.loc[classic['day_n'] == 7, 'retention_rate'].values[0]
classic_day7_users = classic.loc[classic['day_n'] == 7, 'retained_users'].values[0]

compare = pd.DataFrame({
    'type': ['Classic (Day 7)', 'Range (1~7일 내)', 'Rolling (7일 이후)'],
    'retention_rate': [
        classic_day7_rate,
        range_ret_7['retention_rate'].iloc[0],
        rolling_ret['retention_rate'].iloc[0]
    ],
    'retained_users': [
        classic_day7_users,
        range_ret_7['retained_users'].iloc[0],
        rolling_ret['retained_users'].iloc[0]
    ]
})

plt.figure(figsize=(8, 5.5))

colors = sns.color_palette('Blues', len(compare))
barplot = sns.barplot(x='type', y='retention_rate', data=compare, palette=colors)

for i, row in enumerate(compare.itertuples()):
    plt.text(
        i,
        row.retention_rate + (compare['retention_rate'].max() * 0.02),
        f'{row.retention_rate:.1f}%\n({int(row.retained_users):,}명)',
        ha='center',
        va='bottom',
        fontsize=12,
        fontweight='bold'
    )

ax = plt.gca()
ax.spines[['top', 'left', 'right']].set_visible(False)


plt.title('7일 기준 리텐션 유형별 비교', fontsize=15, fontweight='bold')
plt.xlabel('리텐션 산출 방식', fontweight='bold', labelpad=10)
plt.ylabel('리텐션율 (%)', fontweight='bold')
plt.ylim(0, compare['retention_rate'].max() * 1.3)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

- 큰 차이를 보이지 않는 양상.
    - 초기에 **한 번이라도 돌어온 적이 있는 유저는 7일 이후에도 이탈하지 않고** 꾸준히 서비스 이용.
- 첫 레슨 완료 직후 7일 이내 서비스를 이용할 수 있도록 푸시 알림, 독려 메시지 등을 통해 **최대 한 번이라도 더 재진입할 수 있도록 개선** 필요.

# 5. 평일/주말 유저 리텐션 비교

In [ ]:
query = """
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b
            ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s
            ON sc.user_id = s.user_id
           AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc
            ON el.user_id = fc.user_id
           AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl
            ON cl.user_id = fl.user_id
           AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retained_users AS (
        SELECT DISTINCT
            a.user_id
        FROM activation_users a
        JOIN v_events_lesson_view el
            ON a.user_id = el.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <  DATE_ADD(a.activation_time, INTERVAL 8 DAY)
    )

SELECT
    CASE
        WHEN DAYOFWEEK(au.activation_time) IN (1, 7) THEN 'weekend'
        ELSE 'weekday'
    END AS cohort_group,
    COUNT(DISTINCT au.user_id) AS cohort_size,
    COUNT(DISTINCT ru.user_id) AS retained_users,
    ROUND(
        COUNT(DISTINCT ru.user_id) / COUNT(DISTINCT au.user_id) * 100,
        1
    ) AS retention_rate
FROM activation_users au
LEFT JOIN retained_users ru
    ON au.user_id = ru.user_id
GROUP BY cohort_group
ORDER BY retention_rate DESC;
"""

daytype_cohort = pd.read_sql(query, engine)
daytype_cohort

- 평일이 주말보다 약 0.6%p 높은 양상.
    - 레슨 길이가 비교적 짧아 하루 일과 가운데 수강하는 유저가 많은 것으로 예상.

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd

daytype_cohort['not_retained_users'] = daytype_cohort['cohort_size'] - daytype_cohort['retained_users']

contingency_table = daytype_cohort[['retained_users', 'not_retained_users']].values

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"카이제곱 통계량: {chi2:.4f}")
print(f"p-value: {p_value:.4e}")

alpha = 0.05
if p_value < alpha:
    print(f"\n결과: p-value가 {alpha}보다 작으므로, 평일과 주말의 리텐션 차이는 통계적으로 매우 유의미합니다.")
    print("즉, 요일에 따른 리텐션 차이는 우연이 아닙니다.")
else:
    print(f"\n결과: p-value가 {alpha}보다 크므로, 통계적으로 유의미한 차이라고 보기 어렵습니다.")

# 6. 레슨 특징별 재진입 비율

## 6-1. 레슨 제목별 리텐션 비율

In [ ]:
query = """
WITH
    signup AS (
        SELECT s.user_id, MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL GROUP BY s.user_id
    ),

    first_content AS (
        SELECT sc.user_id, MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s ON sc.user_id = s.user_id AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT el.user_id, MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc ON el.user_id = fc.user_id AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT cl.user_id, MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retention_lesson_id AS (
        SELECT
            el.user_id,
            el.lesson_id
        FROM v_events_lesson_view el
        JOIN activation_users a
            ON el.user_id = a.user_id
            AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
            AND el.event_time < DATE_ADD(a.activation_time, INTERVAL 8 DAY)
    ),

    total_retained AS (
        SELECT COUNT(DISTINCT user_id) AS total_users
        FROM retention_lesson_id
    )

SELECT
    rli.lesson_id,
    lm.title,
    COUNT(DISTINCT rli.user_id) AS user_cnt,
    ROUND(
        COUNT(DISTINCT rli.user_id) * 100.0 / (SELECT total_users FROM total_retained), 2
    ) AS user_pct
FROM retention_lesson_id rli
JOIN lesson_dim lm
    ON rli.lesson_id = lm.lesson_id
GROUP BY rli.lesson_id, lm.title
ORDER BY user_cnt DESC;
"""

retention_lesson_df = pd.read_sql(query, engine)
retention_lesson_df.head(10)

- UX/UI 관련 강좌와 실제로 작업해야 하는 실습 프로젝트 관련 강좌의 리텐션 비율이 높은 양상
    - 기초와 실습처럼, 유저가 직접 해 보아야 하는 실습형 레슨이나 기초적인 기반을 다지기 좋은 레슨의 비율을 높이는 것이 필요하다고 생각됨.
    - 혹은 어려운 수준의 강좌들도 기초 수준으로 설명해 줄 수 있도록 개선 필요

## 6-2. 레슨 카테고리별 공급/소비 비율

In [ ]:
query = """
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b
            ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s
            ON sc.user_id = s.user_id
           AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc
            ON el.user_id = fc.user_id
           AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl
            ON cl.user_id = fl.user_id
           AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retention_lessons AS (
        SELECT
            el.user_id,
            el.lesson_id
        FROM v_events_lesson_view el
        JOIN activation_users a
            ON el.user_id = a.user_id
            AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
            AND el.event_time < DATE_ADD(a.activation_time, INTERVAL 8 DAY)
    )

SELECT
    cm.category,
    COUNT(*) AS entry_cnt,
    COUNT(DISTINCT rl.user_id) AS user_cnt,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2
    ) AS entry_pct
FROM retention_lessons rl
JOIN lesson_dim lm
    ON rl.lesson_id = lm.lesson_id
JOIN content_dim cm
    ON lm.content_id = cm.content_id
GROUP BY cm.category
ORDER BY entry_cnt DESC;
"""

retention_cat_df = pd.read_sql(query, engine)
retention_cat_df

In [ ]:
query = """
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s ON sc.user_id = s.user_id AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc ON el.user_id = fc.user_id AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retention_lessons AS (
        SELECT
            el.user_id,
            el.lesson_id
        FROM v_events_lesson_view el
        JOIN activation_users a ON el.user_id = a.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
           AND el.event_time < DATE_ADD(a.activation_time, INTERVAL 8 DAY)
    ),

    supply AS (
        SELECT
            category,
            COUNT(*) AS content_cnt
        FROM content_dim
        WHERE category IS NOT NULL AND category != ''
        GROUP BY category
    ),

    supply_ratio AS (
        SELECT
            category,
            content_cnt,
            content_cnt / SUM(content_cnt) OVER () AS supply_pct
        FROM supply
    ),

    consumption AS (
        SELECT
            cm.category,
            COUNT(*) AS consume_cnt
        FROM retention_lessons rl
        JOIN lesson_dim lm ON rl.lesson_id = lm.lesson_id
        JOIN content_dim cm ON lm.content_id = cm.content_id
        GROUP BY cm.category
    ),

    consumption_ratio AS (
        SELECT
            category,
            consume_cnt,
            consume_cnt / SUM(consume_cnt) OVER () AS consume_pct
        FROM consumption
    )

SELECT
    s.category,
    s.content_cnt,
    ROUND(s.supply_pct * 100, 2) AS supply_pct,
    COALESCE(c.consume_cnt, 0) AS consume_cnt,
    COALESCE(ROUND(c.consume_pct * 100, 2), 0) AS consume_pct,
    COALESCE(ROUND((c.consume_pct - s.supply_pct) * 100, 2), 0) AS lift
FROM supply_ratio s
LEFT JOIN consumption_ratio c ON s.category = c.category
ORDER BY lift DESC;
"""

lift_df = pd.read_sql(query, engine)
lift_df

In [ ]:
plot_df = lift_df.sort_values('lift', ascending=False).copy()

x = np.arange(len(plot_df))
width = 0.35

plt.figure(figsize=(12, 6))


plt.bar(x - width/2, plot_df['supply_pct'], width, color='#bdd8f1', label='공급 비율 (Content %)', alpha=0.8)
plt.bar(x + width/2, plot_df['consume_pct'], width, color='#3667a6', label='리텐션 소비 비율 (Retention %)', alpha=0.9)

for i in range(len(plot_df)):
    plt.text(i + width/2, plot_df['consume_pct'].iloc[i] + 0.5,
             f"{plot_df['consume_pct'].iloc[i]:.1f}%", ha='center', fontsize=9, fontweight='bold')

    lift_val = plot_df['lift'].iloc[i]
    color = '#d90429' if lift_val > 0 else '#2b2d42'
    plt.text(i, max(plot_df['supply_pct'].iloc[i], plot_df['consume_pct'].iloc[i]) + 2,
             f"Lift: {lift_val:+.1f}%", ha='center', color=color, fontsize=10, fontweight='extra bold')

ax = plt.gca()
ax.spines[['top', 'right']].set_visible(False)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.xticks(x, plot_df['category'], rotation=45, ha='right', fontsize=11)
plt.ylabel('비율 (%)', fontweight='bold')
plt.title('카테고리별 공급 vs 리텐션 소비 지표 비교', fontsize=16, fontweight='bold', pad=30)
plt.legend(frameon=False, loc='upper right')

plt.ylim(0, max(plot_df['consume_pct'].max(), plot_df['supply_pct'].max()) + 8)

plt.tight_layout()
plt.show()

In [ ]:
plot_df = lift_df.sort_values('lift', ascending=True).copy()

colors = ['#3667a6' if x > 0 else '#bdd8f1' for x in plot_df['lift']]

plt.figure(figsize=(10, 6))

bars = plt.barh(plot_df['category'], plot_df['lift'], color=colors, alpha=0.9)

plt.axvline(0, color='#1d3557', linewidth=2, linestyle='-')

for i, val in enumerate(plot_df['lift']):
    ha = 'left' if val > 0 else 'right'
    offset = 0.5 if val > 0 else -0.5
    plt.text(val + offset, i, f'{val:+.1f}%p',
             va='center', ha=ha, fontsize=10, fontweight='bold',
             color='#3667a6' if val > 0 else '#8ab6d6')

ax = plt.gca()
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(axis='y', which='both', left=False)

plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.xlabel('Lift (%p) [리텐션 소비 비율 - 공급 비율]', fontweight='bold', labelpad=10)
plt.title('카테고리별 공급 대비 리텐션 초과 소비 지표 (Lift)', fontsize=16, fontweight='bold', pad=25)

limit = max(abs(plot_df['lift'].min()), abs(plot_df['lift'].max())) + 5
plt.xlim(-limit, limit)

plt.tight_layout()
plt.show()

- 개발, 디자인, 영어의 경우 약 5%p 이상으로 공급 대비 소비량이 높은 것을 확인
    - 초기 유입에서와 변함없는 순위를 보이므로, 해당 유저들은 장기적으로 레슨을 수강 및 잔존할 가능성이 높은 것으로 판단.
- 반면, 소비율이 비교적 낮은 마케팅, 자기계발, 비즈니스, 데이터 카테고리의 레슨은 유입부터 개선할 필요성.

# 7. 수강한 레슨 개수

In [ ]:
query = """
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s ON sc.user_id = s.user_id AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc ON el.user_id = fc.user_id AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retention_users AS (
        SELECT
            au.user_id,
            au.activation_time
        FROM activation_users au
        JOIN v_events_lesson_view el ON au.user_id = el.user_id
           AND el.event_time >= DATE_ADD(au.activation_time, INTERVAL 24 HOUR)
           AND el.event_time < DATE_ADD(au.activation_time, INTERVAL 8 DAY)
        GROUP BY au.user_id, au.activation_time
    ),

    lesson_count AS (
        SELECT
            ru.user_id,
            COUNT(cl.event_time) AS lessons_completed
        FROM retention_users ru
        LEFT JOIN v_events_lesson_complete cl ON ru.user_id = cl.user_id
           AND cl.event_time > ru.activation_time
           AND cl.event_time < DATE_ADD(ru.activation_time, INTERVAL 7 DAY)
        GROUP BY ru.user_id
    )

SELECT
    lessons_completed,
    COUNT(DISTINCT user_id) AS user_count,
    ROUND(COUNT(DISTINCT user_id) / SUM(COUNT(DISTINCT user_id)) OVER () * 100, 1) AS pct
FROM lesson_count
GROUP BY 1
ORDER BY 1;
"""

depth_df = pd.read_sql(query, engine)
depth_df

In [ ]:
def categorize(x):
    if x == 0: return '0개'
    elif x <= 10: return '1~10개'
    elif x <= 50: return '11~50개'
    elif x <= 150: return '51~150개'
    elif x <= 300: return '151~300개'
    else: return '301개+'

depth_df['group'] = depth_df['lessons_completed'].apply(categorize)
order = ['0개', '1~10개', '11~50개', '51~150개', '151~300개', '301개+']

depth_plot = depth_df.groupby('group').agg(
    user_count=('user_count', 'sum')
).reindex(order).reset_index()

depth_plot['pct'] = (depth_plot['user_count'] / depth_plot['user_count'].sum() * 100).round(1)


fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('리텐션 유저의 7일간 추가 레슨 완료 수 분포 (Engagement Depth)',
             fontsize=18, fontweight='bold', y=1.05)

bars = axes[0].bar(depth_plot['group'], depth_plot['user_count'],
                   color='#3667a6', alpha=0.85, edgecolor='white', linewidth=1)

axes[0].set_title('구간별 유저 수 분포', fontsize=14, pad=15, fontweight='bold')
axes[0].set_xlabel('레슨 완료 수 구간', fontweight='bold')
axes[0].set_ylabel('유저 수 (명)', fontweight='bold')

axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{int(v):,}'))

for bar in bars:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height + (depth_plot['user_count'].max() * 0.02),
                 f'{int(height):,}명', ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[0].spines[['top', 'right']].set_visible(False)
axes[0].grid(axis='y', linestyle='--', alpha=0.3)

colors = ['#084594', '#2171b5', '#4292c6', '#6baed6', '#9ecae1', '#c6dbef']

explode = [0.05 if p < 5 else 0 for p in depth_plot['pct']]

wedges, texts, autotexts = axes[1].pie(
    depth_plot['pct'],
    labels=depth_plot['group'],
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    pctdistance=0.85,
    explode=explode,
    textprops={'fontsize': 10, 'fontweight': 'bold'}
)

for i, autotext in enumerate(autotexts):
    if i < 2: autotext.set_color('white')

centre_circle = plt.Circle((0,0), 0.70, fc='white')
axes[1].add_artist(centre_circle)

axes[1].set_title('학습 몰입도 구간별 비중', fontsize=14, pad=15, fontweight='bold')

plt.tight_layout()
plt.show()

# 8. 리텐션 유저 연관 질문 클릭 수

In [ ]:
query = """
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s ON sc.user_id = s.user_id AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc ON el.user_id = fc.user_id AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retention_users AS (
        SELECT
            au.user_id,
            au.activation_time
        FROM activation_users au
        JOIN v_events_lesson_view el ON au.user_id = el.user_id
           AND el.event_time >= DATE_ADD(au.activation_time, INTERVAL 24 HOUR)
           AND el.event_time < DATE_ADD(au.activation_time, INTERVAL 8 DAY)
        GROUP BY au.user_id, au.activation_time
    ),

    question_users AS (
        SELECT
            ru.user_id,
            MAX(CASE WHEN q.event_time IS NOT NULL THEN 1 ELSE 0 END) AS has_clicked_question
        FROM retention_users ru
        LEFT JOIN v_events_related_question_click q
            ON ru.user_id = q.user_id
           AND q.event_time >= ru.activation_time
           AND q.event_time < DATE_ADD(ru.activation_time, INTERVAL 8 DAY)
        GROUP BY ru.user_id
    )

SELECT
    CASE WHEN has_clicked_question = 1 THEN '질문 클릭' ELSE '미클릭' END AS question_status,
    COUNT(DISTINCT user_id) AS user_count,
    ROUND(COUNT(DISTINCT user_id) / SUM(COUNT(DISTINCT user_id)) OVER () * 100, 1) AS pct
FROM question_users
GROUP BY 1
ORDER BY user_count DESC;
"""

question_df = pd.read_sql(query, engine)
question_df

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('리텐션 유저의 연관 질문 클릭 행태 분석', fontsize=18, fontweight='bold', y=1.05)

colors = ['#3667a6', '#bdd8f1'] if question_df['question_status'].iloc[0] == '질문 클릭' else ['#bdd8f1', '#3667a6']

bars = axes[0].bar(question_df['question_status'], question_df['user_count'],
                   color=colors, alpha=0.85, edgecolor='white', linewidth=1)

axes[0].set_title('클릭 여부별 유저 수', fontsize=14, pad=15, fontweight='bold')
axes[0].set_xlabel('연관 질문 클릭 여부', fontweight='bold')
axes[0].set_ylabel('유저 수 (명)', fontweight='bold')

for i, (v, p) in enumerate(zip(question_df['user_count'], question_df['pct'])):
    axes[0].text(i, v + (question_df['user_count'].max() * 0.02),
                 f'{v:,}명\n({p}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[0].spines[['top', 'right']].set_visible(False)
axes[0].grid(axis='y', linestyle='--', alpha=0.3)

wedges, texts, autotexts = axes[1].pie(
    question_df['pct'],
    labels=question_df['question_status'],
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    pctdistance=0.75,
    textprops={'fontsize': 11, 'fontweight': 'bold'}
)

centre_circle = plt.Circle((0,0), 0.60, fc='white')
axes[1].add_artist(centre_circle)

axes[1].set_title('질문 클릭 유저 비중 (%)', fontsize=14, pad=15, fontweight='bold')

plt.tight_layout()
plt.show()

# Revenue 연관 분석

In [ ]:
DATE_FMT = '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f'

create_revenue_views_sql = f"""
DROP VIEW IF EXISTS v_events_payment_page_view;
CREATE VIEW v_events_payment_page_view AS
SELECT
    epp.user_id,
    STR_TO_DATE(epp.event_ts, '{DATE_FMT}') AS event_time
FROM events_payment_page_view epp
LEFT JOIN bot_users b ON epp.user_id = b.user_id
WHERE epp.user_id IS NOT NULL AND epp.user_id <> ''
  AND STR_TO_DATE(epp.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;


DROP VIEW IF EXISTS v_events_subscription_complete;
CREATE VIEW v_events_subscription_complete AS
SELECT
    cs.user_id,
    STR_TO_DATE(cs.event_ts, '{DATE_FMT}') AS event_time,
    cs.paid_amount,
    cs.`plan_price`              AS plan_price,
    cs.`discount_amount`  AS coupon_discount_amount,
    cs.`payment_method`                 AS pg_type
FROM events_subscription_complete cs
LEFT JOIN bot_users b ON cs.user_id = b.user_id
WHERE cs.user_id IS NOT NULL AND cs.user_id <> ''
  AND STR_TO_DATE(cs.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;


DROP VIEW IF EXISTS v_events_subscription_renew;
CREATE VIEW v_events_subscription_renew AS
SELECT
    rs.user_id,
    STR_TO_DATE(rs.event_ts, '{DATE_FMT}') AS event_time,
    rs.paid_amount,
    rs.`plan_price`              AS plan_price,
    rs.`discount_amount`  AS coupon_discount_amount,
    rs.`payment_method`                 AS pg_type
FROM events_subscription_renew rs
LEFT JOIN bot_users b ON rs.user_id = b.user_id
WHERE rs.user_id IS NOT NULL AND rs.user_id <> ''
  AND STR_TO_DATE(rs.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;


DROP VIEW IF EXISTS v_events_subscription_resubscribe;
CREATE VIEW v_events_subscription_resubscribe AS
SELECT
    rss.user_id,
    STR_TO_DATE(rss.event_ts, '{DATE_FMT}') AS event_time,
    rss.paid_amount,
    rss.`plan_price`              AS plan_price,
    rss.`discount_amount`  AS coupon_discount_amount,
    rss.`payment_method`                 AS pg_type
FROM events_subscription_resubscribe rss
LEFT JOIN bot_users b ON rss.user_id = b.user_id
WHERE rss.user_id IS NOT NULL AND rss.user_id <> ''
  AND STR_TO_DATE(rss.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;


DROP VIEW IF EXISTS v_events_trial_start;
CREATE VIEW v_events_trial_start AS
SELECT
    ft.user_id,
    STR_TO_DATE(ft.event_ts, '{DATE_FMT}') AS event_time,
    ft.`plan_price` AS plan_price,
    ft.`plan_type`  AS plan_type
FROM events_trial_start ft
LEFT JOIN bot_users b ON ft.user_id = b.user_id
WHERE ft.user_id IS NOT NULL AND ft.user_id <> ''
  AND STR_TO_DATE(ft.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;
"""

execute_many(create_revenue_views_sql)

In [ ]:
DATE_FMT = '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f'

create_events_content_end_view_sql = f"""
DROP VIEW IF EXISTS v_events_content_end;
CREATE VIEW v_events_content_end AS
SELECT
    ec.user_id,
    STR_TO_DATE(ec.event_ts, '{DATE_FMT}') AS event_time
FROM events_content_end ec
LEFT JOIN bot_users b ON ec.user_id = b.user_id
WHERE ec.user_id IS NOT NULL AND ec.user_id <> ''
  AND STR_TO_DATE(ec.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;
"""

execute_many(create_events_content_end_view_sql)

In [ ]:
query = '''
WITH
    signup AS (
        SELECT s.user_id, MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL GROUP BY s.user_id
    ),
    first_content AS (
        SELECT sc.user_id, MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s ON sc.user_id = s.user_id AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),
    first_lesson AS (
        SELECT el.user_id, MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc ON el.user_id = fc.user_id AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),
    activation_users AS (
        SELECT cl.user_id, MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),
    retained_users AS (
        SELECT
            a.user_id,
            a.activation_time,
            MIN(el.event_time) AS retention_time
        FROM activation_users a
        JOIN v_events_lesson_view el
            ON a.user_id = el.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <  DATE_ADD(a.activation_time, INTERVAL 8 DAY)
        GROUP BY a.user_id, a.activation_time
    ),
    additional_completed_users AS (
        SELECT
            r.user_id,
            r.activation_time,
            MIN(cl.event_time) AS additional_complete_time
        FROM retained_users r
        JOIN v_events_lesson_complete cl
            ON r.user_id = cl.user_id
           AND cl.event_time >= r.retention_time
           AND cl.event_time < DATE_ADD(r.activation_time, INTERVAL 8 DAY)
        GROUP BY r.user_id, r.activation_time
    ),
    content_completed_users AS (
        SELECT DISTINCT ac.user_id
        FROM additional_completed_users ac
        JOIN v_events_content_end ec
            ON ac.user_id = ec.user_id
           AND ec.event_time >= ac.additional_complete_time
           AND ec.event_time < DATE_ADD(ac.activation_time, INTERVAL 8 DAY)
    ),

    -- 8. 리텐션 유저를 기준으로 '완강 유저' vs '미완강 유저' 그룹 나누기
    user_groups AS (
        SELECT
            r.user_id,
            r.retention_time,
            CASE
                WHEN cc.user_id IS NOT NULL THEN '완강 유저'
                ELSE '미완강 유저'
            END AS group_type
        FROM retained_users r
        LEFT JOIN content_completed_users cc ON r.user_id = cc.user_id
    ),

    -- 9. 그룹별 '결제 페이지 진입' 추적 (리텐션 발생 이후)
    payment_page_users AS (
        SELECT
            ug.user_id,
            MIN(pp.event_time) AS payment_page_time
        FROM user_groups ug
        JOIN v_events_payment_page_view pp
            ON ug.user_id = pp.user_id
           AND pp.event_time >= ug.retention_time
        GROUP BY ug.user_id
    ),

    paid_events AS (
        SELECT user_id, event_time FROM v_events_subscription_complete
        UNION ALL
        SELECT user_id, event_time FROM v_events_subscription_renew
        UNION ALL
        SELECT user_id, event_time FROM v_events_subscription_resubscribe
    ),

    subscribed_users AS (
        SELECT
            pu.user_id,
            MIN(pe.event_time) AS subscription_time
        FROM payment_page_users pu
        JOIN paid_events pe
            ON pu.user_id = pe.user_id
           AND pe.event_time >= pu.payment_page_time
        GROUP BY pu.user_id
    )

SELECT
    ug.group_type,
    COUNT(DISTINCT ug.user_id) AS total_retained_users,
    COUNT(DISTINCT pu.user_id) AS payment_page_users,
    COUNT(DISTINCT su.user_id) AS subscribed_users,

    ROUND(COUNT(DISTINCT pu.user_id) * 100.0 / COUNT(DISTINCT ug.user_id), 2) AS ret_to_payment_pct,
    ROUND(COUNT(DISTINCT su.user_id) * 100.0 / NULLIF(COUNT(DISTINCT pu.user_id), 0), 2) AS payment_to_sub_pct,
    ROUND(COUNT(DISTINCT su.user_id) * 100.0 / COUNT(DISTINCT ug.user_id), 2) AS total_conversion_pct
FROM user_groups ug
LEFT JOIN payment_page_users pu ON ug.user_id = pu.user_id
LEFT JOIN subscribed_users su ON ug.user_id = su.user_id
GROUP BY ug.group_type;
'''

revenue_funnel_df = pd.read_sql(query, engine)
revenue_funnel_df

In [ ]:
plt.figure(figsize=(10, 6))

metrics = ['ret_to_payment_pct', 'total_conversion_pct']
metric_labels = ['결제창 진입율 (%)', '최종 구독 전환율 (%)']

x = np.arange(len(metric_labels))
width = 0.35

completed_data = revenue_funnel_df[revenue_funnel_df['group_type'] == '완강 유저'][metrics].values.flatten()
incomplete_data = revenue_funnel_df[revenue_funnel_df['group_type'] == '미완강 유저'][metrics].values.flatten()

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, completed_data, width, label='완강 유저', color='#2171b5')
rects2 = ax.bar(x + width/2, incomplete_data, width, label='미완강 유저', color='#bdd8f1')

def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        if pd.notnull(height):
            ax.annotate(f'{height}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 5),
                        textcoords="offset points",
                        ha='center', va='bottom', fontweight='bold', fontsize=12)

autolabel(rects1)
autolabel(rects2)

ax.set_ylabel('전환율 (%)', fontsize=12)
ax.set_title('콘텐츠 완강 여부에 따른 수익화 퍼널 전환율 비교', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
ax.spines[['top', 'right']].set_visible(False)
ax.set_ylim(0, max(max(completed_data), max(incomplete_data)) * 1.3)

plt.tight_layout()
plt.show()

In [ ]:
query = '''
WITH
    signup AS (
        SELECT s.user_id, MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL GROUP BY s.user_id
    ),
    first_content AS (
        SELECT sc.user_id, MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s ON sc.user_id = s.user_id AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),
    first_lesson AS (
        SELECT el.user_id, MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc ON el.user_id = fc.user_id AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),
    activation_users AS (
        SELECT cl.user_id, MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retained_users AS (
        SELECT
            a.user_id
        FROM activation_users a
        JOIN v_events_lesson_view el
            ON a.user_id = el.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <  DATE_ADD(a.activation_time, INTERVAL 8 DAY)
        GROUP BY a.user_id
    ),

    user_groups AS (
        SELECT
            a.user_id,
            a.activation_time,
            CASE
                WHEN r.user_id IS NOT NULL THEN '리텐션 유저 (1~7일)'
                ELSE '비리텐션 유저'
            END AS group_type
        FROM activation_users a
        LEFT JOIN retained_users r ON a.user_id = r.user_id
    ),

    payment_page_users AS (
        SELECT
            ug.user_id,
            MIN(pp.event_time) AS payment_page_time
        FROM user_groups ug
        JOIN v_events_payment_page_view pp
            ON ug.user_id = pp.user_id
           AND pp.event_time >= ug.activation_time
        GROUP BY ug.user_id
    ),

    paid_events AS (
        SELECT user_id, event_time FROM v_events_subscription_complete
        UNION ALL
        SELECT user_id, event_time FROM v_events_subscription_renew
        UNION ALL
        SELECT user_id, event_time FROM v_events_subscription_resubscribe
    ),

    subscribed_users AS (
        SELECT
            pu.user_id,
            MIN(pe.event_time) AS subscription_time
        FROM payment_page_users pu
        JOIN paid_events pe
            ON pu.user_id = pe.user_id
           AND pe.event_time >= pu.payment_page_time
        GROUP BY pu.user_id
    )

SELECT
    ug.group_type,
    COUNT(DISTINCT ug.user_id) AS total_activation_users,
    COUNT(DISTINCT pu.user_id) AS payment_page_users,
    COUNT(DISTINCT su.user_id) AS subscribed_users,

    ROUND(COUNT(DISTINCT pu.user_id) * 100.0 / COUNT(DISTINCT ug.user_id), 2) AS act_to_payment_pct,
    ROUND(COUNT(DISTINCT su.user_id) * 100.0 / NULLIF(COUNT(DISTINCT pu.user_id), 0), 2) AS payment_to_sub_pct,
    ROUND(COUNT(DISTINCT su.user_id) * 100.0 / COUNT(DISTINCT ug.user_id), 2) AS total_conversion_pct
FROM user_groups ug
LEFT JOIN payment_page_users pu ON ug.user_id = pu.user_id
LEFT JOIN subscribed_users su ON ug.user_id = su.user_id
GROUP BY ug.group_type;
'''

retention_vs_churn_df = pd.read_sql(query, engine)
retention_vs_churn_df

In [ ]:
plt.figure(figsize=(10, 6))

metrics = ['act_to_payment_pct', 'total_conversion_pct']
metric_labels = ['결제창 진입율 (%)', '최종 구독 전환율 (%)']

x = np.arange(len(metric_labels))
width = 0.35

retention_data = retention_vs_churn_df[retention_vs_churn_df['group_type'] == '리텐션 유저 (1~7일)'][metrics].values.flatten()
non_retention_data = retention_vs_churn_df[retention_vs_churn_df['group_type'] == '비리텐션 유저'][metrics].values.flatten()

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, retention_data, width, label='리텐션 유저 (1~7일)', color='#08519c')
rects2 = ax.bar(x + width/2, non_retention_data, width, label='비리텐션 유저', color='#9ecae1')

def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        if pd.notnull(height):
            ax.annotate(f'{height}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 5),
                        textcoords="offset points",
                        ha='center', va='bottom', fontweight='bold', fontsize=12)

autolabel(rects1)
autolabel(rects2)

ax.set_ylabel('전환율 (%)', fontsize=12)
ax.set_title('1~7일 리텐션 달성 여부에 따른 수익화 전환율 비교', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
ax.spines[['top', 'right']].set_visible(False)
ax.set_ylim(0, max(max(retention_data), max(non_retention_data)) * 1.3)

plt.tight_layout()
plt.show()